In [1]:
import requests
import pandas as pd
import os
import datetime

## Étape 1 — Extraction / Bronze
Gérer les erreurs lors des appels API : timeout, erreurs HTTP, réponses invalides, etc.

1. Récupérer le dataset des villes marocaines.

In [2]:
ma_url = "https://simplemaps.com/data/ma-cities"
ma_json_url = "https://simplemaps.com/static/data/country-cities/ma/ma.json"

try:
    response = requests.get(url=ma_json_url, timeout=60)
    response.raise_for_status()
    ma_df = pd.DataFrame(data=response.json() or [])
    print(ma_df)
except requests.exceptions.Timeout:
    print("L'API a pris plus d'une minute pour répondre.")
except requests.exceptions.HTTPError as e:
    print(f"Erreur HTTP: {e}")
except requests.exceptions.JSONDecodeError:
    print("Format json invalide.")
# La base des exceptions sourvenues lors d'un request.
except requests.exceptions.RequestException as r:
    print(f"Request Exception: {r}")

                  city      lat       lng  country iso2  \
0           Casablanca  33.5992   -7.6200  Morocco   MA   
1              Tangier  35.7767   -5.8039  Morocco   MA   
2                  Fès  34.0433   -5.0033  Morocco   MA   
3            Marrakech  31.6295   -7.9811  Morocco   MA   
4                 Sale  34.0500   -6.8167  Morocco   MA   
..                 ...      ...       ...      ...  ...   
115        Oulad Yaïch  32.4167   -6.3333  Morocco   MA   
116  Zawyat ech Cheïkh  32.6541   -5.9214  Morocco   MA   
117       Imi-n-Tanout  31.1770   -8.8504  Morocco   MA   
118        Sebt Gzoula  32.1219   -9.0889  Morocco   MA   
119           Tifariti  26.1580  -10.5670  Morocco   MA   

                    admin_name  capital population population_proper  
0            Casablanca-Settat    admin    3950000           3215935  
1    Tanger-Tétouan-Al Hoceïma    admin    1275428           1275428  
2                   Fès-Meknès    admin    1167842           1167842  
3      

2. Utiliser les coordonnées des villes pour interroger l'API Open-Meteo.

In [8]:
meteo_data = []
meteo_url = "https://api.open-meteo.com/v1/forecast"

for lat, lng in ma_df[['lat', 'lng']].to_numpy():
    meteo_params = {
        "latitude": lat,
        "longitude": lng,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "precipitation_probability_max",
            "wind_speed_10m_max",
            "wind_gusts_10m_max"
        ],
        "current": "weather_code",
        "forecast_days": 3
    }
    try:
        response = requests.get(url=meteo_url, params=meteo_params, timeout=300)
        response.raise_for_status()
        meteo_data.append(response.json())
    except requests.exceptions.Timeout:
        print("L'API a pris plus de cinqs minutes pour répondre.")
        break
    except requests.exceptions.HTTPError as e:
        print(f"Erreur HTTP: {e}")
        break
    except requests.exceptions.JSONDecodeError:
        print("Format json invalide.")
        break
    except requests.exceptions.RequestException as r:
        print(f"Request Exception: {r}")
        break

3. Récupérer les prévisions météorologiques quotidiennes des prochains jours.

In [76]:

try:
    meteo_df = pd.DataFrame(data=meteo_data)
except:
    meteo_df = pd.read_csv("bronze/meteo.csv")
meteo_df

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,current_units,current,daily_units,daily
0,33.56250,-7.625000,0.211835,0,GMT,GMT,23.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
1,35.75000,-5.812500,0.336289,0,GMT,GMT,30.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
2,34.06250,-5.000000,0.201464,0,GMT,GMT,389.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
3,31.62500,-8.000000,0.181794,0,GMT,GMT,469.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
4,34.00000,-6.812500,0.179529,0,GMT,GMT,27.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
...,...,...,...,...,...,...,...,...,...,...,...
115,32.43750,-6.312500,1.087189,0,GMT,GMT,503.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
116,32.68750,-5.937500,0.892639,0,GMT,GMT,657.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
117,31.18750,-8.875000,0.195622,0,GMT,GMT,863.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
118,32.12500,-9.062500,0.265956,0,GMT,GMT,173.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."


4. Conserver les données brutes dans bronze/.

In [ ]:
try:
    os.mkdir(path="./bronze")
    ma_df.to_csv(path_or_buf="./bronze/ma.csv", index=False)
    meteo_df.to_csv(path_or_buf="./bronze/meteo.csv", index=False)
except FileExistsError:
    ma_df.to_csv(path_or_buf="./bronze/ma.csv", index=False)
    meteo_df.to_csv(path_or_buf="./bronze/meteo.csv", index=False)
except Exception as e:
    print(e)

In [ ]:
# meteo_params_2 = {
#     "latitude": ma_df["lat"].tolist(),
#     "longitude": ma_df["lng"].tolist(),
#     "daily": [
#         "temperature_2m_max",
#         "temperature_2m_min",
#         "precipitation_sum",
#         "precipitation_probability_max",
#         "wind_speed_10m_max",
#         "wind_gusts_10m_max"
#     ],
#     "current": "weather_code",
#     "forecast_days": 3
# }
# response2 = requests.get(
#     url=meteo_url,
#     params=meteo_params_2,
#     timeout=300
# )
# response2.raise_for_status()
# response2.json()

## Étape 2 — Nettoyage / Silver

1. standardiser les types et les dates.

In [5]:
ma_df.dtypes

city                 str
lat                  str
lng                  str
country              str
iso2                 str
admin_name           str
capital              str
population           str
population_proper    str
dtype: object

In [11]:
ma_df["lat"] = ma_df["lat"].astype(float)
ma_df["lng"] = ma_df["lng"].astype(float)
ma_df["population"] = ma_df["population"].astype(float)
ma_df["population_proper"] = ma_df["population_proper"].astype(float)
ma_df.dtypes

city                     str
lat                  float64
lng                  float64
country                  str
iso2                     str
admin_name               str
capital                  str
population           float64
population_proper    float64
dtype: object

In [38]:
ma_df.columns

Index(['city', 'lat', 'lng', 'country', 'iso2', 'admin_name', 'capital',
       'population', 'population_proper'],
      dtype='str')

In [40]:
ma_df.drop(labels=['iso2','admin_name', 'capital', 'population', 'population_proper'], axis=1, inplace=True)
ma_df.head()

,city,lat,lng,country
0,Casablanca,33.5992,-7.6200,Morocco
1,Tangier,35.7767,-5.8039,Morocco
2,Fès,34.0433,-5.0033,Morocco
3,Marrakech,31.6295,-7.9811,Morocco
4,Sale,34.0500,-6.8167,Morocco


In [77]:
meteo_df.dtypes

latitude                 float64
longitude                float64
generationtime_ms        float64
utc_offset_seconds         int64
timezone                     str
timezone_abbreviation        str
elevation                float64
current_units             object
current                   object
daily_units               object
daily                     object
dtype: object

In [78]:
meteo_df.drop(
    labels=[
        "generationtime_ms", 
        "utc_offset_seconds", 
        "timezone", 
        "timezone_abbreviation",
        "current_units", 
        "daily_units"
    ], 
    axis=1, 
    inplace=True
)
meteo_df

,latitude,longitude,elevation,current,daily
0,33.56250,-7.625000,23.0,"{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
1,35.75000,-5.812500,30.0,"{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
2,34.06250,-5.000000,389.0,"{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
3,31.62500,-8.000000,469.0,"{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
4,34.00000,-6.812500,27.0,"{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
...,...,...,...,...,...
115,32.43750,-6.312500,503.0,"{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
116,32.68750,-5.937500,657.0,"{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
117,31.18750,-8.875000,863.0,"{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
118,32.12500,-9.062500,173.0,"{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."


In [79]:
current_df = pd.DataFrame(
    data=meteo_df["current"].to_list(), 
    index=meteo_df["current"].index
)
current_df.head(1)

,time,interval,weather_code
0,2026-09-17T08:30,900,1


In [46]:
current_df.dtypes

time              str
interval        int64
weather_code    int64
dtype: object

In [80]:
current_df["time"] = pd.to_datetime(current_df.time)
current_df.dtypes

time            datetime64[us]
interval                 int64
weather_code             int64
dtype: object

In [81]:
daily_df = pd.DataFrame(
    data=meteo_df["daily"].to_list(), 
    index=meteo_df["daily"].index
)
daily_df.head(1)

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max
0,"[2026-09-17, 2026-09-18, 2026-09-19]","[25.4, 25.2, 26.7]","[19.9, 18.3, 19.5]","[0.0, 0.0, 0.0]","[0, 0, 0]","[12.2, 11.9, 11.5]","[35.6, 32.4, 32.0]"


In [82]:
daily_df.columns.to_list()

['time',
 'temperature_2m_max',
 'temperature_2m_min',
 'precipitation_sum',
 'precipitation_probability_max',
 'wind_speed_10m_max',
 'wind_gusts_10m_max']

In [83]:
daily_df_exploded = daily_df.explode(daily_df.columns.to_list())

In [51]:
daily_df_exploded.head(4)

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max
0,2026-09-17,25.4,19.9,0.0,0,12.2,35.6
0,2026-09-18,25.2,18.3,0.0,0,11.9,32.4
0,2026-09-19,26.7,19.5,0.0,0,11.5,32.0
1,2026-09-17,26.9,17.6,0.0,0,10.6,25.6


In [52]:
daily_df_exploded.dtypes

time                                str
temperature_2m_max               object
temperature_2m_min               object
precipitation_sum                object
precipitation_probability_max    object
wind_speed_10m_max               object
wind_gusts_10m_max               object
dtype: object

In [53]:
daily_df_exploded.precipitation_probability_max.unique()

array([0, 8, 5, 33, 13, 6, 3, 38, 20, 16, 30, 23, 40, 15, 31, 10, 35, 18,
       14], dtype=object)

In [84]:
# date in iso format 'yyyy-mm-dd'
daily_df_exploded.time = pd.to_datetime(daily_df_exploded.time)
daily_df_exploded.temperature_2m_max = daily_df_exploded.temperature_2m_max.astype(float)
daily_df_exploded.temperature_2m_min = daily_df_exploded.temperature_2m_min.astype(float)
daily_df_exploded.precipitation_sum = daily_df_exploded.precipitation_sum.astype(float)
daily_df_exploded.precipitation_probability_max = (
    daily_df_exploded
    .precipitation_probability_max
    .astype(int)
)
daily_df_exploded.wind_speed_10m_max = daily_df_exploded.wind_speed_10m_max.astype(float)
daily_df_exploded.wind_gusts_10m_max = daily_df_exploded.wind_gusts_10m_max.astype(float)

daily_df_exploded.dtypes

time                             datetime64[us]
temperature_2m_max                      float64
temperature_2m_min                      float64
precipitation_sum                       float64
precipitation_probability_max             int64
wind_speed_10m_max                      float64
wind_gusts_10m_max                      float64
dtype: object

In [85]:
meteo_df.drop(["current", "daily"], axis=1, inplace=True)
meteo_df.columns

Index(['latitude', 'longitude', 'elevation'], dtype='str')

In [86]:
daily_current_df = (
    daily_df_exploded.merge(
        current_df["weather_code"], 
        left_index=True, 
        right_index=True,
        suffixes=("_daily", "_current")
    )
)
daily_current_df.head(10)

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code
0,2026-09-17,25.4,19.9,0.0,0,12.2,35.6,1
0,2026-09-18,25.2,18.3,0.0,0,11.9,32.4,1
0,2026-09-19,26.7,19.5,0.0,0,11.5,32.0,1
1,2026-09-17,26.9,17.6,0.0,0,10.6,25.6,1
1,2026-09-18,28.1,19.0,0.0,0,20.3,47.5,1
1,2026-09-19,27.8,22.5,0.0,0,31.1,74.9,1
2,2026-09-17,29.0,18.1,0.0,0,12.0,32.8,3
2,2026-09-18,31.6,17.9,0.0,0,9.8,26.6,3
2,2026-09-19,34.3,21.2,0.0,0,18.9,47.9,3
3,2026-09-17,30.4,18.4,0.0,0,10.5,31.7,1


In [87]:
daily_current_df.shape

(360, 8)

In [60]:
meteo_df.shape

(120, 5)

In [ ]:
meteo_df = (
    meteo_df.merge(
        daily_current_df,
        left_index=True, 
        right_index=True,
        suffixes=("_meteo", "_daily")
    )
)
meteo_df.shape

(360, 11)

In [89]:
meteo_df.head()

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code
0,33.5625,-7.6250,23.0,2026-09-17,25.4,19.9,0.0,0,12.2,35.6,1
0,33.5625,-7.6250,23.0,2026-09-18,25.2,18.3,0.0,0,11.9,32.4,1
0,33.5625,-7.6250,23.0,2026-09-19,26.7,19.5,0.0,0,11.5,32.0,1
1,35.7500,-5.8125,30.0,2026-09-17,26.9,17.6,0.0,0,10.6,25.6,1
1,35.7500,-5.8125,30.0,2026-09-18,28.1,19.0,0.0,0,20.3,47.5,1


In [63]:
meteo_df.dtypes

latitude                                float64
longitude                               float64
utc_offset_seconds                        int64
timezone_abbreviation                       str
elevation                               float64
time                             datetime64[us]
temperature_2m_max                      float64
temperature_2m_min                      float64
precipitation_sum                       float64
precipitation_probability_max             int64
wind_speed_10m_max                      float64
wind_gusts_10m_max                      float64
weather_code                              int64
dtype: object

2. contrôler la qualité des données.

In [90]:
ma_df[ma_df.duplicated() == True]

,city,lat,lng,country


In [91]:
ma_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   city     120 non-null    str    
 1   lat      120 non-null    float64
 2   lng      120 non-null    float64
 3   country  120 non-null    str    
dtypes: float64(2), str(2)
memory usage: 3.9 KB


In [66]:
ma_df.describe()

,lat,lng
count,120.000000,120.000000
mean,32.722032,-6.837944
std,2.148482,2.408281
min,23.716700,-15.950000
25%,31.568600,-8.357275
50%,33.227200,-6.694650
75%,34.184950,-5.525775
max,35.841400,-1.911400


In [92]:
meteo_df[meteo_df.duplicated() == True]

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code


In [93]:
meteo_df.info()

<class 'pandas.DataFrame'>
Index: 360 entries, 0 to 119
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   latitude                       360 non-null    float64       
 1   longitude                      360 non-null    float64       
 2   elevation                      360 non-null    float64       
 3   time                           360 non-null    datetime64[us]
 4   temperature_2m_max             360 non-null    float64       
 5   temperature_2m_min             360 non-null    float64       
 6   precipitation_sum              360 non-null    float64       
 7   precipitation_probability_max  360 non-null    int64         
 8   wind_speed_10m_max             360 non-null    float64       
 9   wind_gusts_10m_max             360 non-null    float64       
 10  weather_code                   360 non-null    int64         
dtypes: datetime64[us](1), float64(8), i

In [94]:
meteo_df.drop("time", axis=1).describe()

,latitude,longitude,elevation,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code
count,360.000000,360.000000,360.000000,360.000000,360.000000,360.000000,360.000000,360.000000,360.000000,360.000000
mean,32.720886,-6.836718,366.025000,29.232500,18.342500,0.051944,2.513889,18.197778,36.110833,1.825000
std,2.139102,2.402353,375.567592,3.303248,1.978886,0.275856,7.312737,5.618547,9.289556,0.998571
min,23.725834,-15.966187,2.000000,21.100000,13.600000,0.000000,0.000000,8.000000,17.300000,0.000000
25%,31.562500,-8.343750,52.750000,26.675000,16.975000,0.000000,0.000000,14.200000,30.200000,1.000000
50%,33.218750,-6.687500,223.500000,28.800000,18.400000,0.000000,0.000000,16.900000,34.600000,2.000000
75%,34.187500,-5.546875,544.250000,31.500000,19.600000,0.000000,0.000000,20.300000,40.300000,3.000000
max,35.812500,-1.937500,1466.000000,38.400000,26.100000,3.400000,40.000000,41.300000,78.500000,3.000000


3. effectuer la jointure entre les villes et les données météo.

In [95]:
meteo_par_villes_df = meteo_df.merge(ma_df, left_index=True, right_index=True)
meteo_par_villes_df.head()

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code,city,lat,lng,country
0,33.5625,-7.6250,23.0,2026-09-17,25.4,19.9,0.0,0,12.2,35.6,1,Casablanca,33.5992,-7.6200,Morocco
0,33.5625,-7.6250,23.0,2026-09-18,25.2,18.3,0.0,0,11.9,32.4,1,Casablanca,33.5992,-7.6200,Morocco
0,33.5625,-7.6250,23.0,2026-09-19,26.7,19.5,0.0,0,11.5,32.0,1,Casablanca,33.5992,-7.6200,Morocco
1,35.7500,-5.8125,30.0,2026-09-17,26.9,17.6,0.0,0,10.6,25.6,1,Tangier,35.7767,-5.8039,Morocco
1,35.7500,-5.8125,30.0,2026-09-18,28.1,19.0,0.0,0,20.3,47.5,1,Tangier,35.7767,-5.8039,Morocco


In [ ]:
meteo_par_villes_df.reset_index(inplace=True, drop=True)
meteo_par_villes_df.drop(columns=["lat", "lng"], inplace=True)
meteo_par_villes_df.head()

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code,city,country
0,33.5625,-7.6250,23.0,2026-09-17,25.4,19.9,0.0,0,12.2,35.6,1,Casablanca,Morocco
1,33.5625,-7.6250,23.0,2026-09-18,25.2,18.3,0.0,0,11.9,32.4,1,Casablanca,Morocco
2,33.5625,-7.6250,23.0,2026-09-19,26.7,19.5,0.0,0,11.5,32.0,1,Casablanca,Morocco
3,35.7500,-5.8125,30.0,2026-09-17,26.9,17.6,0.0,0,10.6,25.6,1,Tangier,Morocco
4,35.7500,-5.8125,30.0,2026-09-18,28.1,19.0,0.0,0,20.3,47.5,1,Tangier,Morocco


4. stocker les données nettoyées dans silver/.

In [100]:
try:
    os.mkdir(path="./silver")
    meteo_par_villes_df.to_csv(path_or_buf="./silver/cleaned_meteo_per_town.csv", index=False)
except FileExistsError:
    meteo_par_villes_df.to_csv(
        path_or_buf="./silver/cleaned_meteo_per_town.csv", 
        index=False
    )
except Exception as e:
    print(e)

## Étape 3 — Feature Engineering / Gold

1. catégories de température.

In [37]:
meteo_par_villes_df[[column for column in meteo_par_villes_df.columns.to_list() if column.__contains__("temperature")]]

,temperature_2m_max_unit,temperature_2m_min_unit,temperature_2m_max,temperature_2m_min
0,°C,°C,25.4,19.9
1,°C,°C,25.2,18.3
2,°C,°C,26.7,19.5
3,°C,°C,26.9,17.6
4,°C,°C,28.1,19.0
...,...,...,...,...
355,°C,°C,30.0,17.7
356,°C,°C,32.2,18.7
357,°C,°C,31.0,17.8
358,°C,°C,29.4,16.2


2. catégories de précipitations.

3. catégories de vent.

4. date.

5. autres indicateurs pertinents.

6. Weather Risk Score
<br>
Créer un score de risque météorologique de 0 à 100 permettant d'identifier les conditions potentiellement défavorables.

7. Vous devrez justifier :
* les variables utilisées.
* les seuils.
* la méthode de calcul.

8. Charger les données finales dans PostgreSQL.<br>
Le modèle devra permettre de gérer au minimum :
* les villes et leurs coordonnées.
* les prévisions météorologiques.
* le risk_score.<br>
Vous devrez également prévoir une stratégie pour éviter les doublons lors des nouvelles exécutions du pipeline, car les prévisions peuvent être mises à jour.

9. Bonus:
<br>
conserver l'historique des différentes prévisions.

## Étape 4 — Analyse SQL
Réaliser au minimum 5 requêtes SQL répondant à des questions métier.

1. Quelles villes auront les températures les plus élevées ?

2. Quelles villes auront les plus fortes précipitations ?

3. Quelles villes présentent le risque moyen le plus élevé ?

4. Quelles périodes présentent le risque maximal ?

5. Pour chaque ville, quelle période présente le plus grand risque ?

6. Bonus:
sous-requêtes, fonctions de fenêtrage.

## Étape 5 — Dashboard Streamlit
Créer un dashboard connecté à PostgreSQL permettant de visualiser les prévisions et les risques.
<br>
Le dashboard doit permettre de répondre rapidement à la question :
Où et quand faut-il être particulièrement vigilant dans les prochains jours ?

1. KPI (Key Performance Indicator):
* nombre de villes.
* température maximale.
* précipitations maximales.
* nombre de périodes à risque.
* ville présentant le risque le plus élevé.

2. Filtres:
<br>
Permettre de filtrer notamment par : 
* ville. 
* date. 
* période. 
* niveau de risque.

## Étape 6 — Orchestration & automatisation (Airflow)

1. Définir un
DAG Airflow
qui automatise l'ensemble du pipeline :
* Extraction depuis l'API.
* Nettoyage / transformation.
* Feature engineering & chargement dans PostgreSQL.
* (Rafraîchissement des données pour le dashboard).

2. Planifier une exécution automatique (ex. quotidienne) et gérer les échecs (retries).

3. Conteneuriser le projet avec Docker Compose (Postgres + Airflow + Streamlit).

4. Bonus:
* Ajouter du logging structuré et des alertes en cas d'échec du DAG.
* Historique des prévisions.
* Pipeline incrémental.
* Contrôles de qualité.